# real-time ingestion: csv -> spark structured streaming -> postgres

DEM05 lab 2. the same functions as `python -m src.spark_streaming_to_postgres`, one step per section, on
a folder of its own (`data/notebook/`) so it never touches the real
`data/incoming` or `checkpoint/`.

1. setup
2. part 1: generate a few csv files (some rows spoiled on purpose)
3. look at the raw rows the way the stream reads them
4. part 2: cleaning step by step, how many rows each rule drops
5. part 3: stream the folder into postgres once, then check the table
6. what the job logged per batch

## 1. setup

In [1]:
import shutil

import psycopg2
from pyspark.sql import functions as F

from src import cleaning, config
from src.data_generator import write_file
from src.spark_streaming_to_postgres import start_query
from src.spark_session import get_spark

NB_DIR = config.ROOT / "data" / "notebook"
INCOMING = NB_DIR / "incoming"
CHECKPOINT = NB_DIR / "checkpoint"
METRICS = NB_DIR / "metrics"

# fresh start every run: new files, new checkpoint. only the notebook's own folder
shutil.rmtree(NB_DIR, ignore_errors=True)
INCOMING.mkdir(parents=True)

spark = get_spark("events-notebook")
spark.version

C:\Program Files\Python313\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


'4.3.0'

## 2. part 1: generate csv files

4 files of 100 rows, 10% spoiled so the cleaning has something to catch.
`write_file` writes `_name.csv` first and renames it, spark skips names
starting with `_`, so it never sees half a file.

In [2]:
import random

random.seed(5)
for _ in range(4):
    name, bad = write_file(INCOMING, rows=100, bad_rate=0.10)
    print(f"{name}: 100 rows, {bad} spoiled")
sorted(p.name for p in INCOMING.iterdir())

events_20261106_162735_039972.csv: 100 rows, 21 spoiled
events_20261106_162735_042372.csv: 100 rows, 9 spoiled
events_20261106_162735_044457.csv: 100 rows, 11 spoiled
events_20261106_162735_046683.csv: 100 rows, 5 spoiled


['events_20261106_162735_039972.csv',
 'events_20261106_162735_042372.csv',
 'events_20261106_162735_044457.csv',
 'events_20261106_162735_046683.csv']

## 3. the raw rows

same schema as the stream: every column is text. a batch `spark.read` with
`RAW_SCHEMA` shows exactly what each micro-batch will get.

In [3]:
raw = spark.read.schema(cleaning.RAW_SCHEMA).option("header", True).csv(str(INCOMING))
print(f"raw rows: {raw.count()}")
raw.show(5, truncate=False)
raw.printSchema()

raw rows: 400


+------------------------------------+-----------------------+-------+------------+-----------+----------+-----------+-------+--------+
|event_id                            |event_time             |user_id|session_id  |event_type |product_id|category   |price  |quantity|
+------------------------------------+-----------------------+-------+------------+-----------+----------+-----------+-------+--------+
|eac4d424-3bc9-4e4e-8e3f-6af28638e1d7|2026-11-06 14:27:35.043|275    |b58818f98786|view       |1973      |sports     |541.11 |0       |
|c8920d75-8bb2-4e6f-a2f2-02a9f7e36fb1|2026-11-06 14:27:35.043|87     |630eac974267|view       |1429      |sports     |335.84 |0       |
|f9493e8b-8f80-4157-a397-d9d2ff275a38|2026-11-06 14:27:35.043|151    |3b7866638f67|add_to_cart|1631      |fashion    |35.88  |2       |
|d79935cb-68ba-4f38-b597-e04bc3df4fde|2026-11-06 14:27:35.043|456    |a50bfef20958|view       |1528      |home       |116.87 |0       |
|5a6f016c-47e7-4e82-a4e5-ae7816dd7368|2026-11-06

some of the spoiled rows (the generator puts `''`, `-5`, `unknown` or `NaN` in one field):

In [4]:
bad_values = ["", "-5", "unknown", "NaN"]
spoiled = raw.filter(
    F.col("price").isin(bad_values) | F.col("user_id").isin(bad_values)
    | F.col("event_type").isin(bad_values) | F.col("event_time").isin(bad_values)
    | F.col("price").isNull() | F.col("user_id").isNull()
    | F.col("event_type").isNull() | F.col("event_time").isNull()
)
spoiled.select("event_id", "event_time", "user_id", "event_type", "price").show(8, truncate=False)

+------------------------------------+-----------------------+-------+----------+-------+
|event_id                            |event_time             |user_id|event_type|price  |
+------------------------------------+-----------------------+-------+----------+-------+
|c0d9036c-56ac-4d8e-884d-cdbdc1a56f56|2026-11-06 14:27:35.043|21     |view      |unknown|
|62986d6c-c48c-4f9b-acb2-28fcfc854fb3|2026-11-06 14:27:35.043|unknown|view      |52.31  |
|9206b2e2-43fe-4f2b-aa0c-a27edfecebba|NULL                   |53     |view      |1164.63|
|7fca5d32-cd5e-4b92-97e1-947122c0afaf|2026-11-06 14:27:35.043|220    |unknown   |95.56  |
|ccf3d097-9c2c-4f85-aa73-8c1b706ea088|2026-11-06 14:27:35.043|131    |view      |-5     |
|ab6e41e2-c36f-4553-85d9-6a5ac63961eb|2026-11-06 14:27:35.043|306    |NaN       |1148.55|
|8b02ec34-2ab7-4e55-947c-f171d1823f00|NULL                   |30     |view      |70.99  |
|3fabc584-1470-4ca0-963e-93059faa8f1b|2026-11-06 14:27:35.044|268    |NULL      |24.55  |
+---------

## 4. part 2: cleaning step by step

`cleaning.clean` is three steps. running them one by one shows how many rows
each one drops.

In [5]:
typed = cleaning.cast_types(raw)
typed.printSchema()
# try_cast turned the junk into nulls instead of failing
typed.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in ["event_time", "user_id", "price"]]).show()

root
 |-- event_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- session_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)



+----------+-------+-----+
|event_time|user_id|price|
+----------+-------+-----+
|        14|      9|    8|
+----------+-------+-----+



In [6]:
valid = cleaning.drop_bad_rows(typed)
deduped = cleaning.drop_duplicates(valid)
counts = {"raw": raw.count(), "after rules": valid.count(), "after dedup": deduped.count()}
counts

{'raw': 400, 'after rules': 354, 'after dedup': 354}

In [7]:
# which rule caught what. a row can fail more than one, so these can add up to more than the drop
rules = {
    "no event_time": F.col("event_time").isNull(),
    "user_id not > 0": ~(F.coalesce(F.col("user_id") > 0, F.lit(False))),
    "bad event_type": ~F.col("event_type").isin(cleaning.EVENT_TYPES),
    "price not >= 0": ~(F.coalesce(F.col("price") >= 0, F.lit(False))),
}
typed.select([F.sum(F.when(cond, 1).otherwise(0)).alias(name) for name, cond in rules.items()]).show()

+-------------+---------------+--------------+--------------+
|no event_time|user_id not > 0|bad event_type|price not >= 0|
+-------------+---------------+--------------+--------------+
|           14|             10|             6|            12|
+-------------+---------------+--------------+--------------+



In [8]:
# cleaned rows, real types, same columns as the events table
cleaning.clean(raw).show(5, truncate=False)

+------------------------------------+-----------------------+-------+------------+-----------+----------+-----------+-------+--------+
|event_id                            |event_time             |user_id|session_id  |event_type |product_id|category   |price  |quantity|
+------------------------------------+-----------------------+-------+------------+-----------+----------+-----------+-------+--------+
|0339da96-1d86-494c-8d0b-45c3bde46489|2026-11-06 14:27:35.044|112    |fd7c772229a1|view       |1135      |home       |170.01 |0       |
|05467d03-c95d-4a91-8b73-3996ef24c341|2026-11-06 14:27:35.044|446    |2340c82202d3|add_to_cart|1238      |fashion    |230.12 |2       |
|0555e3a9-6b0b-495d-a012-477382c4451c|2026-11-06 14:27:35.044|274    |1f773bf43d93|view       |1993      |electronics|1268.40|0       |
|0595bcab-7ecb-4de6-8226-b96cd9f0d9a6|2026-11-06 14:27:35.044|167    |cc1ddaab1c68|view       |1417      |electronics|302.43 |0       |
|05b0b0f9-8688-4a3d-b7e1-5322b0ede2a8|2026-11-06

## 5. part 3: stream into postgres

`start_query(once=True)` uses the `availableNow` trigger: it takes every file
waiting now, still one file per micro-batch (`MAX_FILES`), and stops by
itself. each batch goes to `events_staging` by jdbc and then one
`INSERT ... ON CONFLICT (event_id) DO NOTHING` moves it into `events`.

first a quick check that postgres is up, so the notebook still runs to the
end when it is not.

In [9]:
def pg_ok() -> bool:
    try:
        psycopg2.connect(connect_timeout=3, **config.PG).close()
        return True
    except psycopg2.OperationalError as e:
        print(f"postgres not reachable on {config.PG['host']}:{config.PG['port']}: {str(e).strip()}")
        return False

PG_UP = pg_ok()
PG_UP

True

In [10]:
if PG_UP:
    query = start_query(spark, input_dir=INCOMING, checkpoint_dir=CHECKPOINT,
                        metrics_dir=METRICS, once=True)
    query.awaitTermination()
    print("batches run:", len(query.recentProgress))
else:
    print("skipped, start postgres and run this section again")

query started: id=aae025b7-2704-48f1-a5d8-aac03ecad3fe


watching C:\Workstation\Amalitech\Week-5\Real-Time-Data-Ingestion\data\notebook\incoming once, up to 1 file(s) per batch


batch 0: 79 rows, 79 inserted, 1.7 s


batch 1: 91 rows, 91 inserted, 1.0 s


batch 2: 89 rows, 89 inserted, 1.0 s


batch 3: 95 rows, 95 inserted, 1.0 s


batches run: 5


In [11]:
# every cleaned event_id should now be in the table, exactly once
if PG_UP:
    ids = [r["event_id"] for r in deduped.select("event_id").collect()]
    with psycopg2.connect(**config.PG) as conn, conn.cursor() as cur:
        cur.execute("SELECT count(*), count(DISTINCT event_id) FROM events WHERE event_id = ANY(%s)", (ids,))
        print("cleaned rows:", len(ids), " in postgres (rows, distinct):", cur.fetchone())

query stopped: exception=None


cleaned rows: 354  in postgres (rows, distinct): (354, 354)


In [12]:
# read it back through spark's jdbc reader, the same driver the sink uses
if PG_UP:
    (spark.read.format("jdbc")
        .option("url", config.JDBC_URL)
        .option("query", "SELECT event_type, count(*) AS n FROM events GROUP BY 1 ORDER BY 2 DESC")
        .option("user", config.PG["user"]).option("password", config.PG["password"])
        .option("driver", "org.postgresql.Driver")
        .load()
        .show())

+-----------+------+
| event_type|     n|
+-----------+------+
|       view|104481|
|add_to_cart| 23429|
|   purchase| 11214|
+-----------+------+



## 6. what the job logged

`write_batch` adds one line per batch to `batches.jsonl` (rows, inserted,
jdbc and merge time). running section 5 again would skip every row: the
checkpoint says the files are done. delete `data/notebook/checkpoint` and
they are read again, and then the merge shows `skipped_duplicates`.

In [13]:
import json

log = METRICS / "batches.jsonl"
[json.loads(line) for line in log.read_text(encoding="utf-8").splitlines()] if log.exists() else "no batches logged (postgres was down)"

[{'batch_id': 0,
  'rows': 79,
  'inserted': 79,
  'skipped_duplicates': 0,
  'jdbc_ms': 1634,
  'merge_ms': 54,
  'at': '2026-11-06T16:27:48'},
 {'batch_id': 1,
  'rows': 91,
  'inserted': 91,
  'skipped_duplicates': 0,
  'jdbc_ms': 980,
  'merge_ms': 53,
  'at': '2026-11-06T16:27:50'},
 {'batch_id': 2,
  'rows': 89,
  'inserted': 89,
  'skipped_duplicates': 0,
  'jdbc_ms': 945,
  'merge_ms': 53,
  'at': '2026-11-06T16:27:53'},
 {'batch_id': 3,
  'rows': 95,
  'inserted': 95,
  'skipped_duplicates': 0,
  'jdbc_ms': 992,
  'merge_ms': 55,
  'at': '2026-11-06T16:27:55'}]

In [14]:
spark.stop()